# 01_eda_cartera

Notebook generado a partir del script Python actualizado del TFM. El código se conserva sin cambios funcionales.

In [4]:
"""
TFM - Segmentación de Clientes con Créditos Activos según Perfil de Riesgo
Etapa 1: Carga de datos y Análisis Exploratorio (EDA)
Autores: Lourdes Flores Mamani / Angel Parra Florecin
Dataset: cartera_creditos (cartera propia) + reporte_crediticio_rcc (RCC-SBS)
Periodo: Noviembre 2024

Versión actualizada: 18/08/2026
- Usa los archivos .txt vigentes.
- Separa registros de créditos activos de clientes únicos.
- Regenera el EDA ampliado desde la misma fuente de 597.127 registros.
- Incluye controles de consistencia para sexo, tipo de préstamo,
  unidad ejecutora y año de apertura.
- Regenera las figuras 1-5 y las figuras ampliadas 7-9.
"""

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

In [6]:
# ─────────────────────────────────────────────
# 0. CONFIGURACIÓN DE RUTAS

In [8]:
# ─────────────────────────────────────────────
DATA_DIR = Path(r"C:\ANGEL\UNIR\TFM 2026 - v2\Data")
OUTPUT_DIR = DATA_DIR / "outputs" / "eda"
OUTPUT_DIR_AMPLIADO = DATA_DIR / "outputs" / "eda_ampliado"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_AMPLIADO.mkdir(parents=True, exist_ok=True)

COD_FILE = DATA_DIR / "cartera_creditos.txt"
RCC_FILE = DATA_DIR / "reporte_crediticio_rcc.txt"

# Variables del EDA inicial. Algunas se conservaron para documentar
# problemas de calidad aunque no formen parte del modelo final.
EDA_VARS = [
    "Dias_Mora",
    "Capital_Vencido",
    "Cuotas_Vencidas",
    "Capital_Judicial",
    "Abono_Promedio",
    "Saldo_Desembolsado",
    "Calificacion_Sbs",
]

NUMERIC_COLS = [
    "Dias_Mora", "Capital_Vencido", "Cuotas_Vencidas", "Capital_Judicial",
    "Abono_Promedio", "Saldo_Desembolsado", "Saldo_Vigente", "Capital_Vigente",
    "Saldo_Mora", "Saldo_Provision", "Monto_Cuota", "Tasa_Interes", "Numero_Dias",
    "Numero_Cuotas", "Cuotas_Pagadas", "Cuotas_Por_Pagar", "Cuotas_Pendientes",
    "Saldo_Amortizacion_Vencida", "Saldo_Interes_Vencido",
    "Abono_Mes1", "Abono_Mes2", "Abono_Mes3",
    "Abono_Mes4", "Abono_Mes5", "Abono_Mes6",
]


def limpiar_categoria(serie, cero_como_sin_identificar=False):
    """Normaliza valores categóricos sin eliminar registros."""
    resultado = serie.fillna("Sin identificar").astype(str).str.strip()
    resultado = resultado.replace(
        ["", "nan", "NaN", "None", "<NA>"],
        "Sin identificar",
    )
    if cero_como_sin_identificar:
        resultado = resultado.replace("0", "Sin identificar")
    return resultado


def buscar_columna(df, candidatos):
    """Devuelve la primera columna disponible entre varios nombres posibles."""
    for columna in candidatos:
        if columna in df.columns:
            return columna
    return None


def validar_total(nombre, df_dist, total_fuente):
    """Comprueba que una distribución conserva todas las filas de la fuente."""
    total_dist = int(df_dist["cantidad"].sum())
    print(f"  Control {nombre:<25}: {total_dist:,} / {total_fuente:,}")
    assert total_dist == total_fuente, (
        f"ERROR en {nombre}: {total_dist:,} != {total_fuente:,}"
    )

In [10]:
# ─────────────────────────────────────────────
# 1. CARGA DE DATOS

In [12]:
# ─────────────────────────────────────────────
print("=" * 70)
print("ETAPA 1 — CARGA Y ANÁLISIS EXPLORATORIO")
print("=" * 70)
print("\n[1/8] Cargando datasets...")

df_cod = pd.read_csv(
    COD_FILE,
    sep=";",
    encoding="latin-1",
    low_memory=False,
    dtype=str,
)

df_rcc = pd.read_csv(
    RCC_FILE,
    sep=";",
    encoding="latin-1",
    low_memory=False,
    dtype=str,
)

TOTAL_REGISTROS = len(df_cod)

print(f"  cartera_creditos       → {df_cod.shape[0]:,} filas × {df_cod.shape[1]} columnas")
print(f"  reporte_crediticio_rcc → {df_rcc.shape[0]:,} filas × {df_rcc.shape[1]} columnas")
print("\n  CONTROL DE FUENTE")
print(f"  Registros cartera      : {TOTAL_REGISTROS:,}")

if "Cliente" in df_cod.columns:
    df_cod["Cliente"] = df_cod["Cliente"].astype(str).str.strip()
    print(f"  Clientes únicos        : {df_cod['Cliente'].nunique():,}")

if "Codigo_Cliente_Sbs" in df_rcc.columns:
    df_rcc["Codigo_Cliente_Sbs"] = df_rcc["Codigo_Cliente_Sbs"].astype(str).str.strip()

ETAPA 1 — CARGA Y ANÁLISIS EXPLORATORIO

[1/8] Cargando datasets...
  cartera_creditos       → 597,127 filas × 88 columnas
  reporte_crediticio_rcc → 1,191,640 filas × 7 columnas

  CONTROL DE FUENTE
  Registros cartera      : 597,127
  Clientes únicos        : 593,638


In [13]:
# ─────────────────────────────────────────────
# 2. CONVERSIÓN DE VARIABLES NUMÉRICAS

In [16]:
# ─────────────────────────────────────────────
print("\n[2/8] Convirtiendo columnas numéricas...")

for col in NUMERIC_COLS:
    if col in df_cod.columns:
        df_cod[col] = pd.to_numeric(df_cod[col], errors="coerce")

for col in ["Saldo", "Condicion_Dias", "Calificacion_Entidad"]:
    if col in df_rcc.columns:
        df_rcc[col] = pd.to_numeric(df_rcc[col], errors="coerce")


[2/8] Convirtiendo columnas numéricas...


In [18]:
# ─────────────────────────────────────────────
# 3. CALIDAD DE DATOS

In [20]:
# ─────────────────────────────────────────────
print("\n[3/8] Analizando calidad de datos...")


def reporte_calidad(df, nombre):
    total = len(df)
    reporte = pd.DataFrame({
        "columna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "nulos": df.isnull().sum().values,
        "pct_nulos": (df.isnull().sum().values / total * 100).round(2),
        "unicos": df.nunique(dropna=True).values,
    }).sort_values("pct_nulos", ascending=False)

    print(f"\n  Dataset: {nombre} ({total:,} registros)")
    con_nulos = reporte[reporte["pct_nulos"] > 0]
    if len(con_nulos) > 0:
        print(con_nulos.to_string(index=False))
    else:
        print("  No se encontraron valores nulos.")
    return reporte


rep_cod = reporte_calidad(df_cod, "cartera_creditos")
rep_rcc = reporte_calidad(df_rcc, "reporte_crediticio_rcc")

rep_cod.to_csv(OUTPUT_DIR / "calidad_cod.csv", index=False, encoding="utf-8-sig")
rep_rcc.to_csv(OUTPUT_DIR / "calidad_rcc.csv", index=False, encoding="utf-8-sig")
rep_cod.to_csv(OUTPUT_DIR_AMPLIADO / "calidad_completa_cod.csv", index=False, encoding="utf-8-sig")
rep_rcc.to_csv(OUTPUT_DIR_AMPLIADO / "calidad_completa_rcc.csv", index=False, encoding="utf-8-sig")


[3/8] Analizando calidad de datos...

  Dataset: cartera_creditos (597,127 registros)
           columna   tipo  nulos  pct_nulos  unicos
     Grupo_Prepago object    804       0.13      12
  Calificacion_Sbs object     53       0.01       6
Codigo_Cliente_Sbs object     53       0.01  593587

  Dataset: reporte_crediticio_rcc (1,191,640 registros)
  No se encontraron valores nulos.


In [22]:
# ─────────────────────────────────────────────
# 4. ESTADÍSTICAS DESCRIPTIVAS

In [24]:
# ─────────────────────────────────────────────
print("\n[4/8] Generando estadísticas descriptivas...")

vars_numericas = [
    v for v in EDA_VARS
    if v in df_cod.columns and v != "Calificacion_Sbs"
]

desc = df_cod[vars_numericas].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
).T
desc["coef_variacion"] = (desc["std"] / desc["mean"]).round(2)
print(desc.round(2).to_string())
desc.to_csv(OUTPUT_DIR / "estadisticas_descriptivas.csv")


[4/8] Generando estadísticas descriptivas...
                       count          mean          std     min           25%           50%           75%          90%           95%          max  coef_variacion
Dias_Mora           597127.0  5.432000e+01       363.17    0.00  0.000000e+00  0.000000e+00  0.000000e+00          0.0  7.500000e+01       7841.0            6.69
Capital_Vencido     597127.0  0.000000e+00         0.00    0.00  0.000000e+00  0.000000e+00  0.000000e+00          0.0  0.000000e+00          0.0             NaN
Cuotas_Vencidas     597127.0  1.320000e+00         6.71    0.00  0.000000e+00  0.000000e+00  0.000000e+00          0.0  3.000000e+00         67.0            5.08
Capital_Judicial    597127.0  1.425984e+08  34206898.25  180.63  1.000000e+08  1.550000e+08  1.699000e+08  169900000.0  1.799000e+08  300000000.0            0.24
Abono_Promedio      597127.0  2.031060e+03      2848.39    0.00  8.119600e+02  1.723290e+03  2.755540e+03       3763.7  4.877380e+03     241839.

In [26]:
# ─────────────────────────────────────────────
# 5. DISTRIBUCIÓN DE CALIFICACIÓN SBS

In [28]:
# ─────────────────────────────────────────────
print("\n[5/8] Distribución de calificación SBS...")

if "Calificacion_Sbs" in df_cod.columns:
    calif_limpia = df_cod["Calificacion_Sbs"].astype(str).str.strip()
    calif_counts = (
        calif_limpia
        .value_counts(dropna=False)
        .rename_axis("calificacion")
        .reset_index(name="cantidad")
    )
    calif_counts["pct"] = (
        calif_counts["cantidad"] / TOTAL_REGISTROS * 100
    ).round(1)
    print(calif_counts.to_string(index=False))
    calif_counts.to_csv(
        OUTPUT_DIR / "dist_calificacion_sbs.csv",
        index=False,
        encoding="utf-8-sig",
    )


[5/8] Distribución de calificación SBS...
  calificacion  cantidad  pct
     2. NORMAL    498602 83.5
    6. PERDIDA     64691 10.8
        3. CPP     18177  3.0
     5. DUDOSO      8227  1.4
 4. DEFICIENTE      7369  1.2
           nan        53  0.0
1. NO DEFINIDO         8  0.0


In [30]:
# ─────────────────────────────────────────────
# 6. EDA AMPLIADO

In [32]:
# ─────────────────────────────────────────────
print("\n[6/8] Generando EDA ampliado...")
print(f"  Fuente utilizada: {TOTAL_REGISTROS:,} registros")

# 6.1 Sexo
COL_SEXO = buscar_columna(df_cod, ["Sexo", "SEXO"])
if COL_SEXO is None:
    raise ValueError("No se encontró la columna Sexo.")

sexo_limpio = limpiar_categoria(df_cod[COL_SEXO], cero_como_sin_identificar=True)
dist_sexo = (
    sexo_limpio
    .value_counts(dropna=False)
    .rename_axis("sexo")
    .reset_index(name="cantidad")
)
dist_sexo["pct"] = (dist_sexo["cantidad"] / TOTAL_REGISTROS * 100).round(2)
validar_total("Sexo", dist_sexo, TOTAL_REGISTROS)
dist_sexo.to_csv(
    OUTPUT_DIR_AMPLIADO / "dist_sexo.csv",
    index=False,
    encoding="utf-8-sig",
)
print(dist_sexo.to_string(index=False))

# 6.2 Tipo de préstamo
COL_TIPO_PRESTAMO = buscar_columna(df_cod, ["Tipo_Prestamo", "Tipo_Préstamo", "TPRESTAMO"])
if COL_TIPO_PRESTAMO is None:
    raise ValueError("No se encontró la columna Tipo_Prestamo.")

tipo_prestamo_limpio = limpiar_categoria(df_cod[COL_TIPO_PRESTAMO])
dist_tipo_prestamo = (
    tipo_prestamo_limpio
    .value_counts(dropna=False)
    .rename_axis("tipo_prestamo")
    .reset_index(name="cantidad")
)
dist_tipo_prestamo["pct"] = (
    dist_tipo_prestamo["cantidad"] / TOTAL_REGISTROS * 100
).round(2)
validar_total("Tipo préstamo", dist_tipo_prestamo, TOTAL_REGISTROS)
dist_tipo_prestamo.to_csv(
    OUTPUT_DIR_AMPLIADO / "dist_tipo_prestamo.csv",
    index=False,
    encoding="utf-8-sig",
)
print(dist_tipo_prestamo.head(15).to_string(index=False))

# 6.3 Unidad ejecutora / distribución institucional
COL_UNIDAD = buscar_columna(
    df_cod,
    [
        "Desc_Unidad_Ejecutora",
        "Unidad_Ejecutora",
        "Nombre_Unidad_Ejecutora",
        "Aunid_Eject",
        "AUNID_EJECT",
        "UNID_EJECT",
    ],
)
if COL_UNIDAD is None:
    raise ValueError("No se encontró la columna de unidad ejecutora.")

print(f"  Columna de unidad ejecutora: {COL_UNIDAD}")
unidad_limpia = limpiar_categoria(df_cod[COL_UNIDAD])
dist_ubicacion = (
    unidad_limpia
    .value_counts(dropna=False)
    .rename_axis("agencia")
    .reset_index(name="cantidad")
)
dist_ubicacion["pct"] = (
    dist_ubicacion["cantidad"] / TOTAL_REGISTROS * 100
).round(2)
validar_total("Unidad ejecutora", dist_ubicacion, TOTAL_REGISTROS)
# Se guarda la distribución completa. La figura usa únicamente el Top 20.
dist_ubicacion.to_csv(
    OUTPUT_DIR_AMPLIADO / "dist_ubicacion.csv",
    index=False,
    encoding="utf-8-sig",
)
print(dist_ubicacion.head(20).to_string(index=False))

# 6.4 Año de apertura
COL_FECHA = buscar_columna(df_cod, ["Fecha_Apertura", "FAPERTURA"])
if COL_FECHA is None:
    raise ValueError("No se encontró la columna Fecha_Apertura.")

fecha_apertura = pd.to_datetime(
    df_cod[COL_FECHA],
    errors="coerce",
    dayfirst=True,
)
anio_texto = fecha_apertura.dt.year.astype("Int64").astype(str).replace(
    "<NA>", "Sin identificar"
)
dist_anio_apertura = (
    anio_texto
    .value_counts(dropna=False)
    .rename_axis("anio")
    .reset_index(name="cantidad")
)
dist_anio_apertura["pct"] = (
    dist_anio_apertura["cantidad"] / TOTAL_REGISTROS * 100
).round(2)
dist_anio_apertura["_orden"] = pd.to_numeric(
    dist_anio_apertura["anio"], errors="coerce"
)
dist_anio_apertura = (
    dist_anio_apertura
    .sort_values("_orden", na_position="last")
    .drop(columns="_orden")
    .reset_index(drop=True)
)
validar_total("Año apertura", dist_anio_apertura, TOTAL_REGISTROS)
dist_anio_apertura.to_csv(
    OUTPUT_DIR_AMPLIADO / "dist_anio_apertura.csv",
    index=False,
    encoding="utf-8-sig",
)
print(dist_anio_apertura.to_string(index=False))


[6/8] Generando EDA ampliado...
  Fuente utilizada: 597,127 registros
  Control Sexo                     : 597,127 / 597,127
           sexo  cantidad   pct
              M    342352 57.33
              F    254766 42.67
Sin identificar         9  0.00
  Control Tipo préstamo            : 597,127 / 597,127
       tipo_prestamo  cantidad   pct
       PREST.MAESTRO    192330 32.21
  PREST.MULTIRED ONP    128138 21.46
  PREST.MULTIRED PNP     84804 14.20
   PRESTAMO MULTIRED     60901 10.20
PREST.MULTIRED MINSA     51927  8.70
MINCETUR/MINISTERIOS     19961  3.34
     MUNICIPALIDADES     11437  1.92
 PREST.UNIVERSIDADES      8522  1.43
   GOBIERNO REGIONAL      8023  1.34
    EJERCITO PERUANO      6957  1.17
PREST.MULTIRED P.JUD      6821  1.14
     PRESTAMO F.A.P.      6147  1.03
 PREST.FF.AA. MARINA      5320  0.89
PREST.MULTIRED BN(8)      4398  0.74
   REFINANCIADO T.C.       883  0.15
  Columna de unidad ejecutora: Desc_Unidad_Ejecutora
  Control Unidad ejecutora         : 597,127 /

In [34]:
# ─────────────────────────────────────────────
# 7. FIGURAS

In [36]:
# ─────────────────────────────────────────────
print("\n[7/8] Generando figuras...")
plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = "#2563EB"

# Figura 1: Calificación SBS
if "Calificacion_Sbs" in df_cod.columns:
    fig, ax = plt.subplots(figsize=(9, 5))
    calif_plot = df_cod["Calificacion_Sbs"].astype(str).str.strip().value_counts().sort_index()
    bars = ax.bar(calif_plot.index, calif_plot.values, color=PALETTE, edgecolor="white")
    ax.bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=9)
    ax.set_title("Distribución de créditos activos por calificación SBS", fontsize=12, pad=12)
    ax.set_xlabel("Calificación SBS")
    ax.set_ylabel("Número de créditos activos")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig01_dist_calificacion_sbs.png", dpi=150)
    plt.close()
    print("  → fig01_dist_calificacion_sbs.png")

# Figura 2: Días de mora
if "Dias_Mora" in df_cod.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    data_mora = df_cod["Dias_Mora"].dropna()
    axes[0].hist(data_mora, bins=60, color=PALETTE, edgecolor="white", alpha=0.85)
    axes[0].set_title("Días de mora — distribución completa")
    axes[0].set_xlabel("Días de mora")
    axes[0].set_ylabel("Frecuencia")
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    data_mora_zoom = data_mora[data_mora <= 365]
    axes[1].hist(data_mora_zoom, bins=60, color="#10B981", edgecolor="white", alpha=0.85)
    axes[1].set_title("Días de mora — zoom 0 a 365 días")
    axes[1].set_xlabel("Días de mora")
    axes[1].set_ylabel("Frecuencia")

    fig.suptitle("Distribución de los días de mora", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig02_dist_ds_mora.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  → fig02_dist_ds_mora.png")

# Figura 3: Boxplots de variables financieras
vars_box = [
    v for v in ["Capital_Vencido", "Saldo_Desembolsado", "Abono_Promedio", "Capital_Judicial"]
    if v in df_cod.columns
]
if vars_box:
    fig, axes = plt.subplots(1, len(vars_box), figsize=(14, 5))
    if len(vars_box) == 1:
        axes = [axes]
    colors = ["#2563EB", "#7C3AED", "#059669", "#DC2626"]
    for ax, var, color in zip(axes, vars_box, colors):
        data_var = df_cod[var].dropna()
        p99 = data_var.quantile(0.99)
        ax.boxplot(
            data_var[data_var <= p99],
            patch_artist=True,
            boxprops=dict(facecolor=color, alpha=0.5),
            medianprops=dict(color="black", linewidth=2),
        )
        ax.set_title(var, fontsize=10)
        ax.set_ylabel("Soles (S/.)")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        ax.set_xticks([])
    fig.suptitle("Distribución de variables financieras clave (percentil ≤99)", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig03_boxplots_vars_financieras.png", dpi=150)
    plt.close()
    print("  → fig03_boxplots_vars_financieras.png")

# Figura 4: Correlación
vars_corr = [v for v in vars_numericas if v in df_cod.columns]
if len(vars_corr) >= 2:
    corr_matrix = df_cod[vars_corr].corr()
    fig, ax = plt.subplots(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        ax=ax,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title("Matriz de correlación — Variables del EDA", fontsize=12, pad=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig04_correlacion_vars_clustering.png", dpi=150)
    plt.close()
    print("  → fig04_correlacion_vars_clustering.png")

# Figura 5: RCC por tipo de crédito
if "Tipo_Credito" in df_rcc.columns and "Saldo" in df_rcc.columns:
    rcc_tipo = df_rcc.groupby("Tipo_Credito")["Saldo"].agg(["count", "sum", "mean"])
    rcc_tipo.columns = ["n_registros", "saldo_total", "saldo_promedio"]
    rcc_tipo = rcc_tipo.sort_values("saldo_total", ascending=False)
    rcc_tipo.to_csv(OUTPUT_DIR / "rcc_por_tipo_credito.csv", encoding="utf-8-sig")

    rcc_top = rcc_tipo.head(10)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(rcc_top.index[::-1], rcc_top["saldo_total"][::-1], color="#7C3AED", alpha=0.8)
    ax.set_title("RCC — Saldo total por tipo de crédito (Top 10)", fontsize=12)
    ax.set_xlabel("Saldo total (S/.)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig05_rcc_saldo_tipo_credito.png", dpi=150)
    plt.close()
    print("  → fig05_rcc_saldo_tipo_credito.png")

# Figura 7: Tipo de préstamo
plot_tipo = dist_tipo_prestamo.head(10).sort_values("cantidad", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_tipo["tipo_prestamo"], plot_tipo["cantidad"], color="#2563EB")
ax.set_title("Distribución de la cartera por tipo de préstamo — Top 10")
ax.set_xlabel("Número de créditos activos")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(OUTPUT_DIR_AMPLIADO / "fig_tipo_prestamo.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → fig_tipo_prestamo.png")

# Figura 8: Unidad ejecutora
plot_ubicacion = dist_ubicacion.head(20).sort_values("cantidad", ascending=True)
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(plot_ubicacion["agencia"], plot_ubicacion["cantidad"], color="#7C3AED")
ax.set_title("Distribución de créditos activos — Top 20 unidades ejecutoras")
ax.set_xlabel("Número de créditos activos")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(OUTPUT_DIR_AMPLIADO / "fig_ubicacion.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → fig_ubicacion.png")

# Figura 9: Año de apertura
anio_fig = dist_anio_apertura[dist_anio_apertura["anio"] != "Sin identificar"].copy()
anio_fig["anio_num"] = pd.to_numeric(anio_fig["anio"], errors="coerce")
anio_fig = anio_fig.dropna(subset=["anio_num"]).sort_values("anio_num")
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(anio_fig["anio"].astype(str), anio_fig["cantidad"], color="#059669")
ax.set_title("Créditos activos por año de apertura")
ax.set_xlabel("Año de apertura")
ax.set_ylabel("Número de créditos activos")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR_AMPLIADO / "fig_anio_apertura.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → fig_anio_apertura.png")


[7/8] Generando figuras...
  → fig01_dist_calificacion_sbs.png
  → fig02_dist_ds_mora.png
  → fig03_boxplots_vars_financieras.png
  → fig04_correlacion_vars_clustering.png
  → fig05_rcc_saldo_tipo_credito.png
  → fig_tipo_prestamo.png
  → fig_ubicacion.png
  → fig_anio_apertura.png


In [38]:
# ─────────────────────────────────────────────
# 8. VALIDACIÓN FINAL Y RESUMEN

In [40]:
# ─────────────────────────────────────────────
print("\n[8/8] Validación final...")
print("\n" + "=" * 70)
print("CONTROLES DE CONSISTENCIA")
print("=" * 70)
print(f"Fuente cartera_creditos       : {TOTAL_REGISTROS:,}")
print(f"dist_sexo.csv                 : {dist_sexo['cantidad'].sum():,}")
print(f"dist_tipo_prestamo.csv        : {dist_tipo_prestamo['cantidad'].sum():,}")
print(f"dist_ubicacion.csv            : {dist_ubicacion['cantidad'].sum():,}")
print(f"dist_anio_apertura.csv        : {dist_anio_apertura['cantidad'].sum():,}")

assert dist_sexo["cantidad"].sum() == TOTAL_REGISTROS
assert dist_tipo_prestamo["cantidad"].sum() == TOTAL_REGISTROS
assert dist_ubicacion["cantidad"].sum() == TOTAL_REGISTROS
assert dist_anio_apertura["cantidad"].sum() == TOTAL_REGISTROS

print("\nTODAS LAS DISTRIBUCIONES COINCIDEN CON LA FUENTE.")

print("\n" + "=" * 70)
print("RESUMEN EDA")
print("=" * 70)
print(f"Total registros de créditos activos : {TOTAL_REGISTROS:,}")
if "Cliente" in df_cod.columns:
    print(f"Clientes únicos                     : {df_cod['Cliente'].nunique():,}")
print(f"Total registros RCC                 : {len(df_rcc):,}")

if "Calificacion_Sbs" in df_cod.columns:
    riesgo_alto = (
        df_cod["Calificacion_Sbs"]
        .astype(str)
        .str.strip()
        .isin(["4. DEFICIENTE", "5. DUDOSO", "6. PERDIDA"])
        .sum()
    )
    pct_riesgo = riesgo_alto / TOTAL_REGISTROS * 100
    print(f"Registros con riesgo alto (Def/Dud/Pérd): {riesgo_alto:,} ({pct_riesgo:.1f}%)")

if "Dias_Mora" in df_cod.columns:
    con_mora = (df_cod["Dias_Mora"] > 0).sum()
    print(f"Registros con mora > 0 días             : {con_mora:,} ({con_mora/TOTAL_REGISTROS*100:.1f}%)")

print(f"\nOutputs EDA:          {OUTPUT_DIR.resolve()}")
print(f"Outputs EDA ampliado: {OUTPUT_DIR_AMPLIADO.resolve()}")
print("=" * 70)
print("EDA COMPLETADO CORRECTAMENTE")
print("=" * 70)


[8/8] Validación final...

CONTROLES DE CONSISTENCIA
Fuente cartera_creditos       : 597,127
dist_sexo.csv                 : 597,127
dist_tipo_prestamo.csv        : 597,127
dist_ubicacion.csv            : 597,127
dist_anio_apertura.csv        : 597,127

TODAS LAS DISTRIBUCIONES COINCIDEN CON LA FUENTE.

RESUMEN EDA
Total registros de créditos activos : 597,127
Clientes únicos                     : 593,638
Total registros RCC                 : 1,191,640
Registros con riesgo alto (Def/Dud/Pérd): 80,287 (13.4%)
Registros con mora > 0 días             : 43,957 (7.4%)

Outputs EDA:          C:\ANGEL\UNIR\TFM 2026 - v2\Data\outputs\eda
Outputs EDA ampliado: C:\ANGEL\UNIR\TFM 2026 - v2\Data\outputs\eda_ampliado
EDA COMPLETADO CORRECTAMENTE
